# 🛡️ SIRCCD — Entrenamiento de Modelo de Anonimización

## Objetivo
Entrenar un detector YOLO para **dos clases** (`face`, `license_plate`) y aplicar
**blur automático** sobre los bounding boxes detectados, garantizando la privacidad
en imágenes de calles antes de almacenarlas en el sistema SIRCCD.

## Datasets utilizados
| Dataset | Clase(s) | Imágenes aprox. | Uso |
|---------|----------|-----------------|-----|
| **WIDER FACE** | `face` | ~16 000 | Rostros en gran variedad de escalas y oclusiones |
| **CCPD** | `license_plate` | ~100 000+ (subsample) | Placas vehiculares, bbox codificado en nombre de archivo |
| **PP4AV** | `face` + `license_plate` | ~3 900 | Escenas urbanas vehiculares — refuerzo del dominio real |

## GPU objetivo
**NVIDIA A100 40 GB** (Colab Pro, High-RAM runtime ~83 GB RAM) — optimizaciones:
`imgsz=1280`, batch auto, `amp=True`, `cache='ram'`, `workers=8`.

## Flujo
1. Instalar dependencias → 2. Configuración → 3. Utilidades → 4-6. Descargar datasets →
7-9. Convertir a YOLO → 10. Unificar/balancear → 11. Estadísticas → 12. `data.yaml` →
13. Modelo → 14. Entrenar → 15-16. Evaluar → 17-18. Demo blur → 19. Resumen

---
## 1. Instalar dependencias y verificar GPU

In [ ]:
# ══════════════════════════════════════════════════════════════
# 1. INSTALACIÓN DE DEPENDENCIAS Y VERIFICACIÓN DE GPU
# ══════════════════════════════════════════════════════════════

!pip install -q ultralytics>=8.4.0 opencv-python-headless gdown pyyaml tqdm matplotlib Pillow scipy

import ultralytics
import torch
import cv2
import numpy as np
import sys

print(f"✅ ultralytics : {ultralytics.__version__}")
print(f"✅ PyTorch     : {torch.__version__}")
print(f"✅ OpenCV      : {cv2.__version__}")
print(f"✅ NumPy       : {np.__version__}")
print(f"✅ Python      : {sys.version.split()[0]}")
print(f"✅ CUDA        : {torch.version.cuda}")

# ── Verificar GPU ──────────────────────────────────────────
if not torch.cuda.is_available():
    raise RuntimeError(
        "❌ No se detectó GPU. Este notebook requiere una GPU NVIDIA.\n"
        "   En Colab: Runtime → Change runtime type → A100 GPU + High RAM."
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_mem / 1024**3

print(f"\n🖥️  GPU detectada: {gpu_name}")
print(f"   VRAM: {vram_gb:.1f} GB")
print(f"   CUDA: {torch.version.cuda}")
try:
    print(f"   cuDNN: {torch.backends.cudnn.version()}")
except Exception:
    pass

is_a100 = "A100" in gpu_name
if is_a100:
    print("\n🚀 A100 detectada — se aplicarán optimizaciones para 40 GB VRAM.")
elif vram_gb >= 40:
    print(f"\n✅ GPU potente detectada ({gpu_name}). Ajustar batch según VRAM.")
else:
    print(f"\n⚠️  GPU pequeña ({gpu_name}, {vram_gb:.0f} GB). Reducir batch/imgsz si hay OOM.")

---
## 2. Configuración global y estructura del proyecto

In [ ]:
# ══════════════════════════════════════════════════════════════
# 2. CONFIGURACIÓN GLOBAL
# ══════════════════════════════════════════════════════════════

import os
import random
from pathlib import Path
from datetime import datetime

# ── Rutas ──────────────────────────────────────────────────
BASE_DIR        = Path("/content/sirccd_anon")
DATASETS_DIR    = BASE_DIR / "datasets"
UNIFIED_DIR     = BASE_DIR / "unified"
RUNS_DIR        = BASE_DIR / "runs"
OUTPUTS_DIR     = BASE_DIR / "outputs"

WIDER_DIR       = DATASETS_DIR / "wider_face"
CCPD_DIR        = DATASETS_DIR / "ccpd"
PP4AV_DIR       = DATASETS_DIR / "pp4av"

# ── Parámetros de entrenamiento ────────────────────────────
EXPERIMENT_NAME = f"anon-yolo11m_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
IMG_SIZE        = 1280          # Alta resolución para rostros/placas pequeñas
BATCH_SIZE      = -1            # -1 = Ultralytics auto-batch (maximiza VRAM)
EPOCHS          = 100
PATIENCE        = 15            # Early stopping
WORKERS         = 8
DEVICE          = "0"
AMP             = True          # FP16 en A100 Tensor Cores
CACHE           = "ram"         # Cargar dataset en RAM (Colab High-RAM ~83 GB)
SEED            = 42

# ── Clases ──────────────────────────────────────────────────
CLASS_NAMES     = ["face", "license_plate"]
NC              = len(CLASS_NAMES)
FACE_ID         = 0
PLATE_ID        = 1

# ── Balanceo ────────────────────────────────────────────────
MAX_FACE_IMAGES   = 12_000      # Limitar WIDER FACE para no dominar
MAX_PLATE_IMAGES  = 12_000      # Limitar CCPD para equilibrar
PP4AV_WEIGHT      = 2           # Duplicar PP4AV (dominio más cercano al caso real)

# ── Crear directorios ──────────────────────────────────────
for split in ("train", "val"):
    (UNIFIED_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (UNIFIED_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

for d in (WIDER_DIR, CCPD_DIR, PP4AV_DIR, RUNS_DIR, OUTPUTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ── Semillas ────────────────────────────────────────────────
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── Google Drive ────────────────────────────────────────────
from google.colab import drive
if not os.path.exists("/content/drive"):
    drive.mount("/content/drive")
    print("✅ Google Drive montado")
else:
    print("✅ Google Drive ya montado")

DRIVE_OUTPUT = Path("/content/drive/MyDrive/SIRCCD_Models/anonymization")
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)

print(f"\n📋 Configuración:")
print(f"   Experimento: {EXPERIMENT_NAME}")
print(f"   Clases:      {CLASS_NAMES}")
print(f"   ImgSize:     {IMG_SIZE}")
print(f"   Epochs:      {EPOCHS}")
print(f"   Batch:       {'auto' if BATCH_SIZE == -1 else BATCH_SIZE}")
print(f"   Workers:     {WORKERS}")
print(f"   AMP:         {AMP}")
print(f"   Cache:       {CACHE}")
print(f"   Seed:        {SEED}")

---
## 3. Funciones utilitarias

In [ ]:
# ══════════════════════════════════════════════════════════════
# 3. FUNCIONES UTILITARIAS REUTILIZABLES
# ══════════════════════════════════════════════════════════════

import shutil
import zipfile
import tarfile
import urllib.request
from tqdm import tqdm
from PIL import Image


def download_file(url: str, dest: str, description: str = "", retries: int = 3):
    """Descarga un archivo con barra de progreso y reintentos."""
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0:
        print(f"  ⏭️  Ya existe: {dest.name}")
        return dest

    for attempt in range(1, retries + 1):
        try:
            print(f"  ⬇️  Descargando {description or dest.name} (intento {attempt}/{retries})...")
            with urllib.request.urlopen(url) as response:
                total = int(response.headers.get("Content-Length", 0))
                with open(dest, "wb") as f, tqdm(
                    total=total, unit="B", unit_scale=True, desc=dest.name
                ) as pbar:
                    while True:
                        chunk = response.read(8192)
                        if not chunk:
                            break
                        f.write(chunk)
                        pbar.update(len(chunk))
            print(f"  ✅ {dest.name} ({dest.stat().st_size / 1e6:.1f} MB)")
            return dest
        except Exception as e:
            print(f"  ⚠️  Error intento {attempt}: {e}")
            if attempt == retries:
                raise
    return dest


def extract_archive(archive_path, dest, description: str = ""):
    """Extrae .zip, .tar.gz o .tar."""
    archive_path = Path(archive_path)
    dest = Path(dest)
    if not archive_path.exists():
        print(f"  ❌ Archivo no encontrado: {archive_path}")
        return
    print(f"  📦 Extrayendo {description or archive_path.name}...")
    if archive_path.suffix == ".zip":
        with zipfile.ZipFile(archive_path, "r") as zf:
            zf.extractall(dest)
    elif archive_path.name.endswith((".tar.gz", ".tgz")):
        with tarfile.open(archive_path, "r:gz") as tf:
            tf.extractall(dest)
    elif archive_path.suffix == ".tar":
        with tarfile.open(archive_path, "r:") as tf:
            tf.extractall(dest)
    else:
        print(f"  ⚠️  Formato no soportado: {archive_path.suffix}")
        return
    print(f"  ✅ Extraído en: {dest}")


def validate_image(path) -> bool:
    """Verifica que una imagen se pueda abrir sin errores."""
    try:
        with Image.open(path) as img:
            img.verify()
        return True
    except Exception:
        return False


def get_image_size(path) -> tuple:
    """Retorna (width, height) de una imagen."""
    with Image.open(path) as img:
        return img.size


def bbox_xyxy_to_yolo(x1, y1, x2, y2, img_w, img_h):
    """[x1, y1, x2, y2] pixeles → [cx, cy, w, h] YOLO normalizado."""
    cx = ((x1 + x2) / 2.0) / img_w
    cy = ((y1 + y2) / 2.0) / img_h
    w  = (x2 - x1) / img_w
    h  = (y2 - y1) / img_h
    return cx, cy, w, h


def bbox_xywh_to_yolo(x, y, w, h, img_w, img_h):
    """[x, y, w, h] pixeles (esquina sup-izq) → YOLO normalizado."""
    cx = (x + w / 2.0) / img_w
    cy = (y + h / 2.0) / img_h
    nw = w / img_w
    nh = h / img_h
    return cx, cy, nw, nh


def is_valid_yolo_bbox(cx, cy, w, h, min_size=0.005):
    """Filtra bboxes inválidos o demasiado pequeños."""
    if w <= 0 or h <= 0:
        return False
    if w < min_size and h < min_size:
        return False
    if not (0 <= cx <= 1 and 0 <= cy <= 1):
        return False
    return True


def clamp_yolo_bbox(cx, cy, w, h):
    """Clampea bbox YOLO a [0, 1]."""
    cx = max(0.0, min(1.0, cx))
    cy = max(0.0, min(1.0, cy))
    w  = min(w, 2 * min(cx, 1 - cx))
    h  = min(h, 2 * min(cy, 1 - cy))
    return cx, cy, w, h


def copy_with_prefix(src: Path, dst_dir: Path, prefix: str) -> Path:
    """Copia archivo con prefijo para evitar colisiones entre datasets."""
    dst = dst_dir / f"{prefix}{src.name}"
    shutil.copy2(src, dst)
    return dst


print("✅ Funciones utilitarias definidas")

---
## 4. Descargar WIDER FACE

WIDER FACE contiene ~32 000 imágenes con ~394 000 rostros anotados en gran variedad
de escalas, poses, oclusiones e iluminaciones—ideal para entrenar un detector de `face`.

**Formato de anotación**: archivo de texto con estructura:
```
imagen_path
num_rostros
x1 y1 w h blur expression illumination invalid occlusion pose
...
```

In [ ]:
# ══════════════════════════════════════════════════════════════
# 4. DESCARGAR WIDER FACE
# ══════════════════════════════════════════════════════════════
# Fuente oficial: http://shuoyang1213.me/WIDERFACE/
# Mirrors en Google Drive (los IDs pueden cambiar; actualizar si fallan).

import gdown

# ── Google Drive IDs (mirrors comunes) ─────────────────────
# Si estos IDs caducan, buscar mirrors actualizados en:
#   https://github.com/ultralytics/yolov5/blob/master/data/widerface.yaml
#   https://huggingface.co/datasets/wider_face

WIDER_TRAIN_GDRIVE = "15hGDLhsx8bLgLcIRD5DhYt5iBxnjNF1M"
WIDER_VAL_GDRIVE   = "1GUCogbp16PMGa39thoMMeWxp7Rp5oM8Q"

# URLs alternativas para las anotaciones
WIDER_ANNOT_URLS = {
    "train": "http://shuoyang1213.me/WIDERFACE/support/bbx_annotation/wider_face_split/wider_face_train_bbx_gt.txt",
    "val":   "http://shuoyang1213.me/WIDERFACE/support/bbx_annotation/wider_face_split/wider_face_val_bbx_gt.txt",
}

wider_train_zip = WIDER_DIR / "WIDER_train.zip"
wider_val_zip   = WIDER_DIR / "WIDER_val.zip"
wider_train_annot = WIDER_DIR / "wider_face_train_bbx_gt.txt"
wider_val_annot   = WIDER_DIR / "wider_face_val_bbx_gt.txt"

# ── Descargar imágenes ─────────────────────────────────────
print("📥 WIDER FACE — Imágenes de entrenamiento...")
if not wider_train_zip.exists():
    try:
        gdown.download(id=WIDER_TRAIN_GDRIVE, output=str(wider_train_zip), quiet=False)
    except Exception as e:
        print(f"  ⚠️  gdown falló: {e}")
        print("  💡 Descarga manual: https://drive.google.com/uc?id=" + WIDER_TRAIN_GDRIVE)
else:
    print(f"  ⏭️  Ya existe: {wider_train_zip.name}")

print("\n📥 WIDER FACE — Imágenes de validación...")
if not wider_val_zip.exists():
    try:
        gdown.download(id=WIDER_VAL_GDRIVE, output=str(wider_val_zip), quiet=False)
    except Exception as e:
        print(f"  ⚠️  gdown falló: {e}")
        print("  💡 Descarga manual: https://drive.google.com/uc?id=" + WIDER_VAL_GDRIVE)
else:
    print(f"  ⏭️  Ya existe: {wider_val_zip.name}")

# ── Descargar anotaciones ──────────────────────────────────
print("\n📥 WIDER FACE — Anotaciones...")
for split, url in WIDER_ANNOT_URLS.items():
    dest = wider_train_annot if split == "train" else wider_val_annot
    if not dest.exists():
        try:
            download_file(url, dest, f"Anotaciones {split}")
        except Exception:
            # Fallback: las anotaciones suelen estar dentro del zip de anotaciones
            print(f"  ⚠️  No se pudo descargar anotaciones {split} directamente.")
            print(f"  💡 Alternativa: descargar 'Face annotations' desde:")
            print(f"     http://shuoyang1213.me/WIDERFACE/")
    else:
        print(f"  ⏭️  Ya existe: {dest.name}")

# ── Extraer ────────────────────────────────────────────────
wider_train_dir = WIDER_DIR / "WIDER_train"
wider_val_dir   = WIDER_DIR / "WIDER_val"

if not wider_train_dir.exists() and wider_train_zip.exists():
    extract_archive(wider_train_zip, WIDER_DIR, "WIDER_train")

if not wider_val_dir.exists() and wider_val_zip.exists():
    extract_archive(wider_val_zip, WIDER_DIR, "WIDER_val")

# ── Validar ────────────────────────────────────────────────
train_imgs = list(wider_train_dir.rglob("*.jpg")) if wider_train_dir.exists() else []
val_imgs   = list(wider_val_dir.rglob("*.jpg")) if wider_val_dir.exists() else []

print(f"\n📊 WIDER FACE:")
print(f"   Train: {len(train_imgs):,} imágenes")
print(f"   Val:   {len(val_imgs):,} imágenes")
print(f"   Anotaciones train: {'✅' if wider_train_annot.exists() else '❌'}")
print(f"   Anotaciones val:   {'✅' if wider_val_annot.exists() else '❌'}")

---
## 5. Descargar CCPD

**CCPD** (Chinese City Parking Dataset) contiene ~300 000+ imágenes de vehículos con
placas. Las coordenadas del bounding box están **codificadas en el nombre del archivo**:

```
area-tilt-bbox-vertices-plate-brightness-blur.jpg
         │
         └─ x1&y1_x2&y2 (top-left, bottom-right en pixeles)
```

Usaremos `ccpd_base` y opcionalmente `ccpd_weather` para variedad de condiciones.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 5. DESCARGAR CCPD
# ══════════════════════════════════════════════════════════════
# Fuente: https://github.com/detectRecog/CCPD
# CCPD2019 en Google Drive. Si el ID caduca, verificar en el repo oficial.

# ── Google Drive IDs ───────────────────────────────────────
# CCPD2019 (placas verdes, ~100K imágenes, ~3.5 GB)
# Si falla, ir a https://github.com/detectRecog/CCPD y buscar links actualizados.
CCPD_GDRIVE_ID = "1rdEsCUcIUaYOVRkx5IMXO0Kl2fS1AD2Z"

ccpd_zip = CCPD_DIR / "CCPD2019.zip"

print("📥 CCPD — Dataset de placas vehiculares...")

if not ccpd_zip.exists():
    try:
        gdown.download(id=CCPD_GDRIVE_ID, output=str(ccpd_zip), quiet=False)
        print(f"  ✅ Descargado: {ccpd_zip.name}")
    except Exception as e:
        print(f"  ⚠️  Descarga automática falló: {e}")
        print(f"  💡 Alternativas:")
        print(f"     1. Descargar desde: https://github.com/detectRecog/CCPD")
        print(f"     2. Buscar 'CCPD dataset' en Kaggle")
        print(f"     3. Subir manualmente a: {ccpd_zip}")
else:
    print(f"  ⏭️  Ya existe: {ccpd_zip.name}")

# ── Extraer ────────────────────────────────────────────────
ccpd_extracted = CCPD_DIR / "CCPD2019"
if not ccpd_extracted.exists() and ccpd_zip.exists():
    extract_archive(ccpd_zip, CCPD_DIR, "CCPD2019")

# ── Localizar carpetas de imágenes ─────────────────────────
#    CCPD tiene subcarpetas: ccpd_base, ccpd_weather, ccpd_blur, etc.
#    Usamos ccpd_base (el principal) + ccpd_weather (condiciones variadas)

ccpd_image_dirs = []
for subdir_name in ["ccpd_base", "ccpd_weather", "ccpd_challenge", "ccpd_np"]:
    # Buscar en posibles rutas (la estructura varía según la versión)
    for candidate in [
        CCPD_DIR / "CCPD2019" / subdir_name,
        CCPD_DIR / subdir_name,
        CCPD_DIR / "CCPD2019" / "ccpd" / subdir_name,
    ]:
        if candidate.exists():
            ccpd_image_dirs.append(candidate)
            break

# Si no encontramos subcarpetas, buscar recursivamente
if not ccpd_image_dirs:
    all_jpgs = list(CCPD_DIR.rglob("*.jpg"))
    if all_jpgs:
        # Usar el directorio padre más común
        parents = set(p.parent for p in all_jpgs[:100])
        ccpd_image_dirs = list(parents)
        print(f"  📂 Encontrados {len(all_jpgs)} JPGs en {len(parents)} carpetas")

# ── Contar imágenes ────────────────────────────────────────
total_ccpd = 0
for d in ccpd_image_dirs:
    n = len(list(d.glob("*.jpg")))
    total_ccpd += n
    print(f"   {d.name}: {n:,} imágenes")

print(f"\n📊 CCPD total: {total_ccpd:,} imágenes")
if total_ccpd == 0:
    print("  ⚠️  No se encontraron imágenes CCPD.")
    print("  💡 Descarga manual necesaria: https://github.com/detectRecog/CCPD")

---
## 6. Descargar PP4AV (opcional — refuerzo de dominio)

**PP4AV** (Privacy Preserving for Autonomous Vehicles) contiene escenas urbanas con
rostros y placas anotados en contexto de conducción—el dominio más cercano a nuestro
caso de uso real. Las anotaciones están en formato COCO JSON.

> **Nota**: Si la descarga falla (el dataset es académico y puede cambiar de host),
> el notebook continúa sin PP4AV. WIDER FACE + CCPD son suficientes para un buen modelo.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 6. DESCARGAR PP4AV (OPCIONAL)
# ══════════════════════════════════════════════════════════════
# Paper: "PP4AV: A Benchmark for Privacy-Preserving Autonomous Driving"
# Repo:  https://github.com/2eme-686/pp4av
#
# Si el repo cambia de ubicación, actualizar PP4AV_REPO_URL.
# Si no se puede descargar, el notebook continúa sin PP4AV.

PP4AV_REPO_URL = "https://github.com/2eme-686/pp4av.git"
PP4AV_AVAILABLE = False

print("📥 PP4AV — Dataset de anonimización vehicular (opcional)...")

pp4av_repo_dir = PP4AV_DIR / "pp4av"

if pp4av_repo_dir.exists() and any(pp4av_repo_dir.iterdir()):
    print(f"  ⏭️  Ya existe: {pp4av_repo_dir}")
    PP4AV_AVAILABLE = True
else:
    try:
        import subprocess
        result = subprocess.run(
            ["git", "clone", "--depth", "1", PP4AV_REPO_URL, str(pp4av_repo_dir)],
            capture_output=True, text=True, timeout=120
        )
        if result.returncode == 0:
            print(f"  ✅ Clonado PP4AV")
            PP4AV_AVAILABLE = True
        else:
            print(f"  ⚠️  git clone falló: {result.stderr[:200]}")
    except Exception as e:
        print(f"  ⚠️  No se pudo descargar PP4AV: {e}")

if PP4AV_AVAILABLE:
    # Buscar imágenes y anotaciones
    pp4av_images = list(pp4av_repo_dir.rglob("*.jpg")) + list(pp4av_repo_dir.rglob("*.png"))
    pp4av_jsons  = list(pp4av_repo_dir.rglob("*.json"))
    print(f"\n📊 PP4AV:")
    print(f"   Imágenes:    {len(pp4av_images):,}")
    print(f"   JSONs:       {len(pp4av_jsons)}")
    if not pp4av_images:
        print("  ⚠️  No se encontraron imágenes. PP4AV puede requerir descarga separada.")
        PP4AV_AVAILABLE = False
else:
    print("\n⏭️  PP4AV no disponible — continuando con WIDER FACE + CCPD.")
    print("   El modelo funcionará bien sin PP4AV.")

---
## 7. Convertir WIDER FACE a formato YOLO

Lee las anotaciones oficiales, convierte cada bounding box de rostro a formato YOLO
(`class_id cx cy w h` normalizado) y copia las imágenes al directorio unificado
con prefijo `wf_` para evitar colisiones con otros datasets.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 7. CONVERTIR WIDER FACE → YOLO
# ══════════════════════════════════════════════════════════════
#
# Formato de anotación WIDER FACE:
#   linea 1:  ruta_relativa_imagen (ej: "0--Parade/0_Parade_xxx.jpg")
#   linea 2:  numero_de_rostros (int)
#   lineas 3+: x1 y1 w h blur expression illumination invalid occlusion pose
#   (si num_rostros=0, hay UNA línea con "0 0 0 0 0 0 0 0 0 0")

def convert_wider_face(annot_path, images_root, split, max_images=None):
    """
    Convierte anotaciones WIDER FACE al formato YOLO y copia al dataset unificado.

    Args:
        annot_path:  Ruta al archivo .txt de anotaciones
        images_root: Directorio raíz de imágenes (ej: WIDER_train/images)
        split:       "train" o "val"
        max_images:  Límite de imágenes a procesar (None = sin límite)
    """
    if not Path(annot_path).exists():
        print(f"  ❌ No se encontró: {annot_path}")
        return 0, 0, 0

    dst_img_dir = UNIFIED_DIR / "images" / split
    dst_lbl_dir = UNIFIED_DIR / "labels" / split

    with open(annot_path, "r") as f:
        lines = f.readlines()

    idx = 0
    total_images = 0
    total_boxes  = 0
    skipped_boxes = 0

    while idx < len(lines):
        if max_images and total_images >= max_images:
            break

        # Leer ruta de imagen
        img_rel = lines[idx].strip()
        idx += 1
        if not img_rel or not img_rel.endswith(".jpg"):
            continue

        # Leer número de rostros
        num_faces = int(lines[idx].strip())
        idx += 1

        # Construir ruta completa de la imagen
        img_path = Path(images_root) / img_rel
        if not img_path.exists():
            # Intentar con subdirectorio "images"
            img_path = Path(images_root) / "images" / img_rel
            if not img_path.exists():
                idx += max(1, num_faces)
                continue

        # Obtener dimensiones
        try:
            img_w, img_h = get_image_size(img_path)
        except Exception:
            idx += max(1, num_faces)
            continue

        # Parsear bounding boxes
        yolo_lines = []
        faces_to_read = max(1, num_faces)  # Al menos 1 línea incluso si num_faces=0

        for _ in range(faces_to_read):
            if idx >= len(lines):
                break
            parts = lines[idx].strip().split()
            idx += 1

            if num_faces == 0:
                continue  # Línea dummy

            if len(parts) < 4:
                continue

            x1, y1, w, h = float(parts[0]), float(parts[1]), float(parts[2]), float(parts[3])

            # Columna "invalid" es parts[7] si existe
            is_invalid = int(parts[7]) if len(parts) > 7 else 0
            if is_invalid:
                skipped_boxes += 1
                continue

            # Filtrar cajas con dimensiones inválidas
            if w <= 0 or h <= 0:
                skipped_boxes += 1
                continue

            # Convertir a YOLO
            cx, cy, nw, nh = bbox_xywh_to_yolo(x1, y1, w, h, img_w, img_h)
            cx, cy, nw, nh = clamp_yolo_bbox(cx, cy, nw, nh)

            if is_valid_yolo_bbox(cx, cy, nw, nh, min_size=0.003):
                yolo_lines.append(f"{FACE_ID} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")
                total_boxes += 1
            else:
                skipped_boxes += 1

        # Guardar solo si hay al menos un bbox válido
        if yolo_lines:
            prefix = "wf_"
            safe_name = img_rel.replace("/", "_").replace("\\", "_")
            dst_img = dst_img_dir / f"{prefix}{safe_name}"
            dst_lbl = dst_lbl_dir / f"{prefix}{Path(safe_name).stem}.txt"

            shutil.copy2(img_path, dst_img)
            with open(dst_lbl, "w") as lf:
                lf.write("\n".join(yolo_lines) + "\n")

            total_images += 1

    return total_images, total_boxes, skipped_boxes


# ── Ejecutar conversión ────────────────────────────────────
print("🔄 Convirtiendo WIDER FACE train...")
wf_train_imgs, wf_train_boxes, wf_train_skip = convert_wider_face(
    wider_train_annot,
    WIDER_DIR / "WIDER_train",
    split="train",
    max_images=MAX_FACE_IMAGES
)

print("🔄 Convirtiendo WIDER FACE val...")
wf_val_imgs, wf_val_boxes, wf_val_skip = convert_wider_face(
    wider_val_annot,
    WIDER_DIR / "WIDER_val",
    split="val",
    max_images=MAX_FACE_IMAGES // 5  # ~20% para val
)

print(f"\n📊 WIDER FACE → YOLO:")
print(f"   Train: {wf_train_imgs:,} imgs, {wf_train_boxes:,} faces ({wf_train_skip:,} descartados)")
print(f"   Val:   {wf_val_imgs:,} imgs, {wf_val_boxes:,} faces ({wf_val_skip:,} descartados)")

---
## 8. Convertir CCPD a formato YOLO

En CCPD, el bounding box de la placa está codificado en el **nombre del archivo**.
El tercer campo (separado por `-`) contiene `x1&y1_x2&y2` en píxeles absolutos.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 8. CONVERTIR CCPD → YOLO
# ══════════════════════════════════════════════════════════════
#
# Formato del nombre de archivo CCPD:
#   025-95_113-154&383_386&473-386&473_177&454_154&383_363&402-0_0_22_27_27_33_16-37-15.jpg
#   │      │     │              │                              │                │   │
#   area  tilt  BBOX           vertices                       plate_chars     bright blur
#
# BBOX (campo 2, 0-indexed): "154&383_386&473" → (x1=154, y1=383), (x2=386, y2=473)

def parse_ccpd_filename(filename: str):
    """
    Extrae el bounding box del nombre de archivo CCPD.
    Retorna (x1, y1, x2, y2) en píxeles o None si el formato es inválido.
    """
    try:
        stem = Path(filename).stem
        parts = stem.split("-")
        if len(parts) < 4:
            return None

        # Campo 2: bbox "x1&y1_x2&y2"
        bbox_str = parts[2]
        coords = bbox_str.split("_")
        if len(coords) != 2:
            return None

        x1_y1 = coords[0].split("&")
        x2_y2 = coords[1].split("&")

        x1 = int(x1_y1[0])
        y1 = int(x1_y1[1])
        x2 = int(x2_y2[0])
        y2 = int(x2_y2[1])

        # Validar que sea un bbox razonable
        if x2 <= x1 or y2 <= y1:
            return None

        return x1, y1, x2, y2
    except (ValueError, IndexError):
        return None


def convert_ccpd(image_dirs, split_ratio=0.9, max_images=None):
    """
    Convierte imágenes CCPD al formato YOLO.

    Args:
        image_dirs: Lista de directorios con imágenes CCPD
        split_ratio: Fracción para train (resto para val)
        max_images: Máximo de imágenes a procesar
    """
    # Recolectar todas las imágenes
    all_images = []
    for d in image_dirs:
        all_images.extend(sorted(d.glob("*.jpg")))

    if not all_images:
        print("  ❌ No se encontraron imágenes CCPD.")
        return 0, 0, 0, 0

    # Subsamplear si hay demasiadas
    if max_images and len(all_images) > max_images:
        random.shuffle(all_images)
        all_images = all_images[:max_images]

    # Split train/val
    random.shuffle(all_images)
    split_idx = int(len(all_images) * split_ratio)
    splits = {
        "train": all_images[:split_idx],
        "val":   all_images[split_idx:],
    }

    total_imgs  = 0
    total_boxes = 0
    skipped     = 0

    for split, images in splits.items():
        dst_img_dir = UNIFIED_DIR / "images" / split
        dst_lbl_dir = UNIFIED_DIR / "labels" / split

        for img_path in tqdm(images, desc=f"CCPD → {split}", leave=False):
            bbox = parse_ccpd_filename(img_path.name)
            if bbox is None:
                skipped += 1
                continue

            x1, y1, x2, y2 = bbox

            # Obtener dimensiones reales
            try:
                img_w, img_h = get_image_size(img_path)
            except Exception:
                skipped += 1
                continue

            # Convertir a YOLO
            cx, cy, w, h = bbox_xyxy_to_yolo(x1, y1, x2, y2, img_w, img_h)
            cx, cy, w, h = clamp_yolo_bbox(cx, cy, w, h)

            if not is_valid_yolo_bbox(cx, cy, w, h):
                skipped += 1
                continue

            # Copiar imagen
            dst_img = dst_img_dir / f"ccpd_{img_path.name}"
            shutil.copy2(img_path, dst_img)

            # Escribir label
            dst_lbl = dst_lbl_dir / f"ccpd_{img_path.stem}.txt"
            with open(dst_lbl, "w") as f:
                f.write(f"{PLATE_ID} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}\n")

            total_imgs  += 1
            total_boxes += 1

    train_count = len(splits["train"])
    val_count   = len(splits["val"])
    return total_imgs, total_boxes, skipped, val_count


# ── Ejecutar ────────────────────────────────────────────────
print("🔄 Convirtiendo CCPD...")
ccpd_imgs, ccpd_boxes, ccpd_skip, ccpd_val = convert_ccpd(
    ccpd_image_dirs,
    split_ratio=0.9,
    max_images=MAX_PLATE_IMAGES
)

print(f"\n📊 CCPD → YOLO:")
print(f"   Total:   {ccpd_imgs:,} imgs, {ccpd_boxes:,} plates ({ccpd_skip:,} descartados)")
print(f"   Val:     {ccpd_val:,} imgs")

---
## 9. Convertir PP4AV a formato YOLO (si disponible)

PP4AV usa anotaciones COCO JSON. Mapeamos sus categorías a nuestras clases:
- `face`, `person_face`, `head` → `0 (face)`
- `license_plate`, `plate`, `LP`, `number_plate` → `1 (license_plate)`

> Si PP4AV no está disponible, esta celda se salta automáticamente.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 9. CONVERTIR PP4AV → YOLO (SI DISPONIBLE)
# ══════════════════════════════════════════════════════════════

import json as _json

# Mapeo flexible de categorías PP4AV → nuestras clases
PP4AV_FACE_NAMES  = {"face", "person_face", "head", "Face", "FACE"}
PP4AV_PLATE_NAMES = {"license_plate", "plate", "lp", "LP", "number_plate",
                     "license plate", "License_plate", "License Plate"}

pp4av_stats = {"train": {"face": 0, "plate": 0, "imgs": 0},
               "val":   {"face": 0, "plate": 0, "imgs": 0}}

if PP4AV_AVAILABLE:
    print("🔄 Convirtiendo PP4AV...")

    # Buscar anotaciones COCO JSON
    annot_files = list(pp4av_repo_dir.rglob("*.json"))
    print(f"  Encontrados {len(annot_files)} archivos JSON")

    for annot_file in annot_files:
        try:
            with open(annot_file) as f:
                coco = _json.load(f)
        except Exception:
            continue

        # Verificar que sea formato COCO
        if "images" not in coco or "annotations" not in coco:
            continue

        # Determinar split basado en el nombre del archivo/carpeta
        annot_str = str(annot_file).lower()
        if "val" in annot_str or "test" in annot_str:
            split = "val"
        else:
            split = "train"

        # Mapear categorías
        cat_map = {}
        for cat in coco.get("categories", []):
            name = cat["name"]
            if name in PP4AV_FACE_NAMES or "face" in name.lower():
                cat_map[cat["id"]] = FACE_ID
            elif name in PP4AV_PLATE_NAMES or "plate" in name.lower():
                cat_map[cat["id"]] = PLATE_ID

        if not cat_map:
            print(f"  ⚠️  Sin categorías mapeables en {annot_file.name}")
            cats_found = [c["name"] for c in coco.get("categories", [])]
            print(f"      Categorías encontradas: {cats_found}")
            # Intentar mapeo por posición si hay exactamente 2 categorías
            cats = coco.get("categories", [])
            if len(cats) == 2:
                cat_map[cats[0]["id"]] = FACE_ID
                cat_map[cats[1]["id"]] = PLATE_ID
                print(f"      Asignando: {cats[0]['name']}→face, {cats[1]['name']}→plate")

        # Índice: image_id → info
        img_info = {img["id"]: img for img in coco["images"]}

        # Agrupar anotaciones por imagen
        from collections import defaultdict
        anns_by_img = defaultdict(list)
        for ann in coco["annotations"]:
            if ann["category_id"] in cat_map:
                anns_by_img[ann["image_id"]].append(ann)

        # Directorio de imágenes (relativo al JSON)
        img_base_dir = annot_file.parent

        for img_id, anns in anns_by_img.items():
            info = img_info.get(img_id)
            if info is None:
                continue

            # Buscar imagen
            img_name = info["file_name"]
            img_path = None
            for candidate in [
                img_base_dir / img_name,
                img_base_dir / "images" / img_name,
                pp4av_repo_dir / img_name,
                pp4av_repo_dir / "images" / img_name,
            ]:
                if candidate.exists():
                    img_path = candidate
                    break

            if img_path is None:
                continue

            img_w = info.get("width")
            img_h = info.get("height")
            if not img_w or not img_h:
                try:
                    img_w, img_h = get_image_size(img_path)
                except Exception:
                    continue

            # Convertir anotaciones
            yolo_lines = []
            for ann in anns:
                class_id = cat_map[ann["category_id"]]
                bbox = ann["bbox"]  # COCO: [x, y, w, h]

                cx, cy, w, h = bbox_xywh_to_yolo(bbox[0], bbox[1], bbox[2], bbox[3], img_w, img_h)
                cx, cy, w, h = clamp_yolo_bbox(cx, cy, w, h)

                if is_valid_yolo_bbox(cx, cy, w, h):
                    yolo_lines.append(f"{class_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
                    if class_id == FACE_ID:
                        pp4av_stats[split]["face"] += 1
                    else:
                        pp4av_stats[split]["plate"] += 1

            if yolo_lines:
                # Duplicar PP4AV_WEIGHT veces (dominio más cercano al caso real)
                for dup in range(PP4AV_WEIGHT):
                    suffix = f"_d{dup}" if dup > 0 else ""
                    dst_img = UNIFIED_DIR / "images" / split / f"pp4av_{Path(img_name).stem}{suffix}{img_path.suffix}"
                    dst_lbl = UNIFIED_DIR / "labels" / split / f"pp4av_{Path(img_name).stem}{suffix}.txt"
                    shutil.copy2(img_path, dst_img)
                    with open(dst_lbl, "w") as f:
                        f.write("\n".join(yolo_lines) + "\n")

                pp4av_stats[split]["imgs"] += 1

    print(f"\n📊 PP4AV → YOLO:")
    for split in ("train", "val"):
        s = pp4av_stats[split]
        print(f"   {split}: {s['imgs']} imgs, {s['face']} faces, {s['plate']} plates")
    print(f"   (×{PP4AV_WEIGHT} duplicación aplicada)")
else:
    print("⏭️  PP4AV no disponible — saltando conversión.")

---
## 10. Unificar, validar y balancear el dataset combinado

In [ ]:
# ══════════════════════════════════════════════════════════════
# 10. UNIFICAR, VALIDAR Y BALANCEAR
# ══════════════════════════════════════════════════════════════

from collections import Counter

def validate_unified_dataset():
    """
    Verifica integridad: cada imagen debe tener label y viceversa.
    Elimina huérfanos y archivos corruptos.
    """
    removed = 0
    for split in ("train", "val"):
        img_dir = UNIFIED_DIR / "images" / split
        lbl_dir = UNIFIED_DIR / "labels" / split

        # Imágenes sin label
        for img in img_dir.iterdir():
            if img.suffix.lower() not in (".jpg", ".jpeg", ".png"):
                continue
            lbl = lbl_dir / f"{img.stem}.txt"
            if not lbl.exists():
                img.unlink()
                removed += 1

        # Labels sin imagen
        for lbl in lbl_dir.iterdir():
            if lbl.suffix != ".txt":
                continue
            found = False
            for ext in (".jpg", ".jpeg", ".png"):
                if (img_dir / f"{lbl.stem}{ext}").exists():
                    found = True
                    break
            if not found:
                lbl.unlink()
                removed += 1

    return removed


def count_annotations():
    """Cuenta anotaciones por clase y split."""
    stats = {}
    for split in ("train", "val"):
        lbl_dir = UNIFIED_DIR / "labels" / split
        class_counts = Counter()
        img_count = 0
        total_anns = 0

        for lbl in lbl_dir.glob("*.txt"):
            with open(lbl) as f:
                lines = [l.strip() for l in f if l.strip()]
            if lines:
                img_count += 1
                total_anns += len(lines)
                for line in lines:
                    cls_id = int(line.split()[0])
                    class_counts[cls_id] += 1

        stats[split] = {
            "images": img_count,
            "face": class_counts.get(FACE_ID, 0),
            "plate": class_counts.get(PLATE_ID, 0),
            "total_anns": total_anns,
        }
    return stats


# ── Validar ────────────────────────────────────────────────
print("🔍 Validando dataset unificado...")
removed = validate_unified_dataset()
print(f"   Huérfanos eliminados: {removed}")

# ── Estadísticas ────────────────────────────────────────────
stats = count_annotations()

print(f"\n📊 Dataset unificado:")
print(f"{'':>6} {'Imágenes':>10} {'face':>10} {'plate':>10} {'Total ann':>12}")
print(f"{'─'*50}")
for split in ("train", "val"):
    s = stats[split]
    print(f"{split:>6} {s['images']:>10,} {s['face']:>10,} {s['plate']:>10,} {s['total_anns']:>12,}")

total_imgs = stats["train"]["images"] + stats["val"]["images"]
total_face = stats["train"]["face"] + stats["val"]["face"]
total_plate = stats["train"]["plate"] + stats["val"]["plate"]
print(f"{'TOTAL':>6} {total_imgs:>10,} {total_face:>10,} {total_plate:>10,} {total_face + total_plate:>12,}")

# ── Ratio de balance ────────────────────────────────────────
if total_plate > 0:
    ratio = total_face / total_plate
    print(f"\n⚖️  Ratio face:plate = {ratio:.1f}:1")
    if ratio > 3:
        print(f"   ⚠️  Desbalance alto. Considerar aumentar MAX_PLATE_IMAGES o reducir MAX_FACE_IMAGES.")
    elif ratio < 0.33:
        print(f"   ⚠️  Desbalance alto (más plates que faces).")
    else:
        print(f"   ✅ Balance aceptable.")

---
## 11. Estadísticas y visualización

Inspección visual de imágenes con bounding boxes para confirmar que las conversiones
son correctas antes de entrenar.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 11. Estadísticas del dataset y visualización
# ══════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from collections import Counter

def dataset_statistics(base_dir: Path):
    """Calcula estadísticas detalladas del dataset unificado."""
    stats = {"train": Counter(), "val": Counter()}
    img_counts = {"train": 0, "val": 0}
    bbox_sizes = {"train": [], "val": []}

    for split in ["train", "val"]:
        label_dir = base_dir / "labels" / split
        image_dir = base_dir / "images" / split
        if not label_dir.exists():
            continue
        img_counts[split] = len(list(image_dir.glob("*.*")))
        for lf in sorted(label_dir.glob("*.txt")):
            for line in lf.read_text().strip().splitlines():
                parts = line.strip().split()
                if len(parts) >= 5:
                    cls_id = int(parts[0])
                    w, h = float(parts[3]), float(parts[4])
                    stats[split][cls_id] += 1
                    bbox_sizes[split].append((cls_id, w, h))

    return stats, img_counts, bbox_sizes


stats, img_counts, bbox_sizes = dataset_statistics(DATASET_DIR)

CLASS_NAMES_ANON = {0: "face", 1: "license_plate"}

print("=" * 60)
print("ESTADÍSTICAS DEL DATASET UNIFICADO")
print("=" * 60)

for split in ["train", "val"]:
    print(f"\n{'─' * 40}")
    print(f"  {split.upper()}")
    print(f"{'─' * 40}")
    print(f"  Imágenes: {img_counts[split]:,}")
    total_ann = sum(stats[split].values())
    print(f"  Anotaciones totales: {total_ann:,}")
    for cls_id in sorted(stats[split].keys()):
        name = CLASS_NAMES_ANON.get(cls_id, f"clase_{cls_id}")
        count = stats[split][cls_id]
        pct = (count / total_ann * 100) if total_ann > 0 else 0
        print(f"    {name}: {count:,} ({pct:.1f}%)")

# ── Gráfico de barras por clase ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, split in zip(axes, ["train", "val"]):
    classes = sorted(stats[split].keys())
    names = [CLASS_NAMES_ANON.get(c, f"cls_{c}") for c in classes]
    counts = [stats[split][c] for c in classes]
    colors = ["#3B82F6", "#F59E0B"]
    bars = ax.bar(names, counts, color=colors[:len(names)])
    ax.set_title(f"{split.upper()} — Distribución de clases", fontsize=13, fontweight="bold")
    ax.set_ylabel("Anotaciones")
    for bar, count in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                f"{count:,}", ha="center", va="bottom", fontsize=11)

plt.tight_layout()
plt.savefig(str(DATASET_DIR / "class_distribution.png"), dpi=150)
plt.show()
print(f"\n✅ Gráfico guardado en {DATASET_DIR / 'class_distribution.png'}")

# ── Visualizar muestras con bounding boxes ──
def show_samples_with_boxes(base_dir: Path, split: str = "train", n: int = 8):
    """Dibuja bounding boxes sobre imágenes de muestra."""
    from PIL import Image
    image_dir = base_dir / "images" / split
    label_dir = base_dir / "labels" / split

    image_files = sorted(image_dir.glob("*.*"))
    if not image_files:
        print(f"⚠️ No hay imágenes en {split}")
        return

    indices = np.random.choice(len(image_files), min(n, len(image_files)), replace=False)
    samples = [image_files[i] for i in indices]

    cols = 4
    rows = (len(samples) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(20, 5 * rows))
    if rows == 1:
        axes = [axes] if cols == 1 else axes
    axes_flat = np.array(axes).flatten()

    colors_box = {"0": "#3B82F6", "1": "#F59E0B"}  # azul=face, amarillo=plate

    for idx, (ax, img_path) in enumerate(zip(axes_flat, samples)):
        img = Image.open(img_path).convert("RGB")
        w_img, h_img = img.size
        ax.imshow(img)

        label_path = label_dir / (img_path.stem + ".txt")
        if label_path.exists():
            for line in label_path.read_text().strip().splitlines():
                parts = line.strip().split()
                if len(parts) >= 5:
                    cls_id, cx, cy, bw, bh = parts[0], *[float(x) for x in parts[1:5]]
                    x1 = (cx - bw / 2) * w_img
                    y1 = (cy - bh / 2) * h_img
                    rect_w = bw * w_img
                    rect_h = bh * h_img
                    color = colors_box.get(cls_id, "#EF4444")
                    rect = patches.Rectangle(
                        (x1, y1), rect_w, rect_h,
                        linewidth=2, edgecolor=color, facecolor="none"
                    )
                    ax.add_patch(rect)
                    label_name = CLASS_NAMES_ANON.get(int(cls_id), cls_id)
                    ax.text(x1, y1 - 3, label_name, color=color,
                            fontsize=8, fontweight="bold",
                            bbox=dict(boxstyle="round,pad=0.2", facecolor="black", alpha=0.7))

        ax.set_title(img_path.name[:30], fontsize=9)
        ax.axis("off")

    for ax in axes_flat[len(samples):]:
        ax.axis("off")

    plt.suptitle(f"Muestras del split '{split}' con bounding boxes", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()

show_samples_with_boxes(DATASET_DIR, "train", 8)
show_samples_with_boxes(DATASET_DIR, "val", 4)

---
## 12. Generar `data.yaml`

Archivo de configuración YOLO con las rutas absolutas y las clases del modelo de anonimización.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 12. Generar data.yaml
# ══════════════════════════════════════════════════════════════
import yaml

data_yaml = {
    "path": str(DATASET_DIR),
    "train": "images/train",
    "val": "images/val",
    "nc": NUM_CLASSES,
    "names": CLASS_NAMES,
}

yaml_path = DATASET_DIR / "data.yaml"
with open(yaml_path, "w") as f:
    yaml.dump(data_yaml, f, default_flow_style=False, sort_keys=False)

print("✅ data.yaml generado:")
print(f"   📁 {yaml_path}")
print()
print(yaml_path.read_text())

---
## 13. Selección de modelo y optimización para A100

Se usa **YOLO11m** para balance entre velocidad y precisión. Si no está disponible,
se hace fallback a YOLOv8m.

| Param | YOLO11m | YOLOv8m |
|---|---|---|
| Params | ~20M | ~25.9M |
| GFLOPs | ~68 | ~78.9 |
| mAP (COCO) | ~51.5 | ~50.2 |

La A100 con 40GB VRAM permite `batch=-1` (auto-batch) e `imgsz=1280`.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 13. Selección de modelo y optimización A100
# ══════════════════════════════════════════════════════════════
from ultralytics import YOLO

# Intentar YOLO11m, fallback a YOLOv8m
try:
    model = YOLO("yolo11m.pt")
    MODEL_NAME = "yolo11m"
    print("✅ YOLO11m cargado correctamente")
except Exception as e:
    print(f"⚠️ YOLO11m no disponible ({e}), usando YOLOv8m...")
    model = YOLO("yolov8m.pt")
    MODEL_NAME = "yolov8m"
    print("✅ YOLOv8m cargado como fallback")

# Verificar arquitectura
print(f"\n📋 Modelo: {MODEL_NAME}")
total_params = sum(p.numel() for p in model.model.parameters())
print(f"   Parámetros: {total_params:,}")
print(f"   Tipo: {model.type}")

# Tabla de hiperparámetros
print(f"""
{'=' * 60}
CONFIGURACIÓN DE ENTRENAMIENTO (A100 40GB)
{'=' * 60}
  Modelo base:     {MODEL_NAME}.pt
  Imagen:          {IMG_SIZE}px
  Epochs:          {EPOCHS}
  Batch:           auto (A100 40GB → ~16-24 a 1280px)
  Workers:         {WORKERS}
  AMP:             True (FP16 nativo en A100)
  Optimizer:       auto (AdamW)
  LR inicial:      0.01
  LR final:        0.01 (cosine decay)
  Patience:        15 epochs (early stopping)
  Save period:     10 epochs
  Augmentations:   mosaic=1.0, mixup=0.1, copy_paste=0.1
  Seed:            {SEED}
{'=' * 60}
""")

---
## 14. Entrenamiento

Entrenamiento completo con configuración optimizada para A100 (40 GB VRAM, High-RAM runtime).
Si ocurre OOM, se reduce automáticamente `imgsz` a 960px y se reintenta.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 14. Entrenamiento
# ══════════════════════════════════════════════════════════════
import time
import torch

# Limpiar caché de GPU antes de entrenar
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    print(f"🔧 GPU: {torch.cuda.get_device_name(0)}")
    total_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"   VRAM total: {total_mem:.1f} GB")

PROJECT_NAME = "sirccd-anonymization"
RUN_NAME = f"{MODEL_NAME}-{IMG_SIZE}-e{EPOCHS}"

print(f"\n🚀 Iniciando entrenamiento: {RUN_NAME}")
print(f"   data.yaml: {DATASET_DIR / 'data.yaml'}")
start_time = time.time()

try:
    results = model.train(
        data=str(DATASET_DIR / "data.yaml"),
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=-1,              # auto-batch para A100 40GB
        workers=WORKERS,
        device=0,
        project=str(BASE_DIR / "runs"),
        name=RUN_NAME,
        exist_ok=True,
        pretrained=True,
        optimizer="auto",
        lr0=0.01,
        lrf=0.01,
        momentum=0.937,
        weight_decay=0.0005,
        warmup_epochs=3.0,
        warmup_momentum=0.8,
        warmup_bias_lr=0.1,
        amp=True,              # FP16 nativo en A100
        patience=15,
        save_period=10,
        seed=SEED,
        # Augmentations
        mosaic=1.0,
        mixup=0.1,
        copy_paste=0.1,
        degrees=10.0,
        translate=0.1,
        scale=0.5,
        fliplr=0.5,
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,
        # Logging
        verbose=True,
        plots=True,
    )
    elapsed = time.time() - start_time
    print(f"\n✅ Entrenamiento completado en {elapsed/60:.1f} minutos")

except RuntimeError as e:
    if "out of memory" in str(e).lower():
        print(f"\n⚠️ OOM con imgsz={IMG_SIZE}, reintentando con 960px...")
        torch.cuda.empty_cache()
        results = model.train(
            data=str(DATASET_DIR / "data.yaml"),
            epochs=EPOCHS,
            imgsz=960,
            batch=-1,
            workers=WORKERS,
            device=0,
            project=str(BASE_DIR / "runs"),
            name=f"{RUN_NAME}-960",
            exist_ok=True,
            pretrained=True,
            optimizer="auto",
            lr0=0.01,
            lrf=0.01,
            amp=True,
            patience=15,
            save_period=10,
            seed=SEED,
            mosaic=1.0,
            mixup=0.1,
            copy_paste=0.1,
            degrees=10.0,
            scale=0.5,
            verbose=True,
            plots=True,
        )
        elapsed = time.time() - start_time
        print(f"\n✅ Entrenamiento (960px) completado en {elapsed/60:.1f} minutos")
    else:
        raise e

# Mostrar uso de memoria
if torch.cuda.is_available():
    peak_mem = torch.cuda.max_memory_allocated() / 1e9
    print(f"📊 Pico de VRAM: {peak_mem:.2f} GB")

---
## 15. Evaluación del modelo

Métricas detalladas por clase: Precision, Recall, mAP50, mAP50-95.
Incluye matriz de confusión y curvas P-R.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 15. Evaluación del modelo
# ══════════════════════════════════════════════════════════════
from pathlib import Path
from IPython.display import Image, display

# Buscar el directorio de resultados del entrenamiento
runs_dir = BASE_DIR / "runs"
run_dirs = sorted(runs_dir.glob(f"{RUN_NAME}*"), key=lambda p: p.stat().st_mtime)
if run_dirs:
    TRAIN_DIR = run_dirs[-1]
else:
    TRAIN_DIR = runs_dir / RUN_NAME
print(f"📁 Directorio de resultados: {TRAIN_DIR}")

# Cargar el mejor modelo
best_model_path = TRAIN_DIR / "weights" / "best.pt"
if best_model_path.exists():
    best_model = YOLO(str(best_model_path))
    print(f"✅ Mejor modelo cargado: {best_model_path}")
    print(f"   Tamaño: {best_model_path.stat().st_size / 1e6:.1f} MB")
else:
    print(f"⚠️ No se encontró best.pt en {TRAIN_DIR / 'weights'}")
    print("   Usando el modelo del entrenamiento...")
    best_model = model

# Validación formal
print("\n🔍 Ejecutando validación en el split val...")
val_results = best_model.val(
    data=str(DATASET_DIR / "data.yaml"),
    imgsz=IMG_SIZE,
    batch=-1,
    device=0,
    verbose=True,
    plots=True,
)

# Tabla de métricas por clase
print(f"""
{'=' * 70}
RESULTADOS DE EVALUACIÓN
{'=' * 70}
{'Clase':<20} {'Precision':>10} {'Recall':>10} {'mAP50':>10} {'mAP50-95':>10}
{'─' * 70}""")

class_names = val_results.names
box = val_results.box

# Métricas globales
print(f"{'ALL':<20} {box.mp:>10.4f} {box.mr:>10.4f} {box.map50:>10.4f} {box.map:>10.4f}")

# Métricas por clase
for i, name in class_names.items():
    if i < len(box.p):
        print(f"{name:<20} {box.p[i]:>10.4f} {box.r[i]:>10.4f} {box.ap50[i]:>10.4f} {box.ap[i]:>10.4f}")

print(f"{'─' * 70}")

# Mostrar gráficos generados automáticamente
plot_files = [
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "PR_curve.png",
    "F1_curve.png",
    "results.png",
]

for pf in plot_files:
    plot_path = TRAIN_DIR / pf
    if plot_path.exists():
        print(f"\n📊 {pf}:")
        display(Image(filename=str(plot_path), width=800))
    else:
        print(f"   ⚠️ {pf} no encontrado")

---
## 16. Visualizar predicciones

Comparar imágenes originales vs predicciones del modelo en el split de validación.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 16. Visualizar predicciones
# ══════════════════════════════════════════════════════════════
from PIL import Image as PILImage

val_images_dir = DATASET_DIR / "images" / "val"
val_images = sorted(val_images_dir.glob("*.*"))

n_samples = min(8, len(val_images))
indices = np.random.choice(len(val_images), n_samples, replace=False)
sample_images = [val_images[i] for i in indices]

fig, axes = plt.subplots(n_samples, 2, figsize=(20, 5 * n_samples))
if n_samples == 1:
    axes = axes.reshape(1, -1)

for row, img_path in enumerate(sample_images):
    # Original
    img = PILImage.open(img_path).convert("RGB")
    axes[row, 0].imshow(img)
    axes[row, 0].set_title(f"Original: {img_path.name[:35]}", fontsize=10)
    axes[row, 0].axis("off")

    # Predicción
    pred_results = best_model.predict(
        str(img_path), imgsz=IMG_SIZE, conf=0.25, device=0, verbose=False
    )
    pred_img = pred_results[0].plot(
        conf=True, labels=True, line_width=2
    )
    # pred_img viene en BGR (OpenCV), convertir a RGB
    axes[row, 1].imshow(pred_img[:, :, ::-1])

    # Contar detecciones
    n_det = len(pred_results[0].boxes)
    det_classes = []
    if n_det > 0:
        for box in pred_results[0].boxes:
            cls_id = int(box.cls[0])
            conf = float(box.conf[0])
            name = CLASS_NAMES.get(cls_id, f"cls_{cls_id}")
            det_classes.append(f"{name}:{conf:.2f}")

    det_str = ", ".join(det_classes) if det_classes else "Ninguna"
    axes[row, 1].set_title(f"Predicción ({n_det} det): {det_str}", fontsize=10)
    axes[row, 1].axis("off")

plt.suptitle("Comparación: Original vs Predicción", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 17. Pipeline de anonimización (Blur)

Función principal que aplica **GaussianBlur** sobre las detecciones de rostros y placas.
Configurable: intensidad del blur, pixelación alternativa, margen expandido.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 17. Pipeline de anonimización
# ══════════════════════════════════════════════════════════════
import cv2

def anonymize_image(
    image_path: str,
    model: YOLO,
    conf: float = 0.25,
    imgsz: int = 1280,
    blur_intensity: int = 51,
    margin: float = 0.1,
    method: str = "gaussian",  # "gaussian" o "pixelate"
    device: int = 0,
) -> tuple:
    """
    Detecta rostros y placas, aplica blur/pixelación.

    Args:
        image_path: Ruta a la imagen
        model: Modelo YOLO cargado
        conf: Umbral de confianza
        imgsz: Tamaño de inferencia
        blur_intensity: Kernel size para GaussianBlur (debe ser impar)
        margin: Margen extra alrededor del bbox (0.1 = 10%)
        method: "gaussian" para GaussianBlur, "pixelate" para pixelación
        device: GPU device

    Returns:
        (imagen_anonimizada, detecciones_count, detecciones_detalle)
    """
    img = cv2.imread(str(image_path))
    if img is None:
        raise ValueError(f"No se pudo leer la imagen: {image_path}")

    h, w = img.shape[:2]
    img_anon = img.copy()

    # Inferencia
    results = model.predict(str(image_path), imgsz=imgsz, conf=conf, device=device, verbose=False)
    detections = []

    if len(results[0].boxes) == 0:
        return img_anon, 0, []

    for box in results[0].boxes:
        cls_id = int(box.cls[0])
        confidence = float(box.conf[0])
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)

        # Expandir bbox con margen
        bw = x2 - x1
        bh = y2 - y1
        mx = int(bw * margin)
        my = int(bh * margin)
        x1 = max(0, x1 - mx)
        y1 = max(0, y1 - my)
        x2 = min(w, x2 + mx)
        y2 = min(h, y2 + my)

        # Aplicar anonimización
        roi = img_anon[y1:y2, x1:x2]
        if roi.size == 0:
            continue

        if method == "gaussian":
            # Asegurar kernel impar
            k = blur_intensity if blur_intensity % 2 == 1 else blur_intensity + 1
            blurred = cv2.GaussianBlur(roi, (k, k), 0)
            img_anon[y1:y2, x1:x2] = blurred
        elif method == "pixelate":
            # Reducir y escalar de vuelta
            small_h = max(1, (y2 - y1) // 10)
            small_w = max(1, (x2 - x1) // 10)
            small = cv2.resize(roi, (small_w, small_h), interpolation=cv2.INTER_LINEAR)
            pixelated = cv2.resize(small, (x2 - x1, y2 - y1), interpolation=cv2.INTER_NEAREST)
            img_anon[y1:y2, x1:x2] = pixelated

        cls_name = CLASS_NAMES.get(cls_id, f"cls_{cls_id}")
        detections.append({
            "class": cls_name,
            "confidence": confidence,
            "bbox": [x1, y1, x2, y2],
        })

    return img_anon, len(detections), detections


# ── Demo: anonimizar muestras del val set ──
print("🔒 Demo de anonimización en imágenes de validación\n")

demo_images = sorted(val_images_dir.glob("*.*"))
demo_indices = np.random.choice(len(demo_images), min(4, len(demo_images)), replace=False)
demo_samples = [demo_images[i] for i in demo_indices]

fig, axes = plt.subplots(len(demo_samples), 3, figsize=(24, 6 * len(demo_samples)))
if len(demo_samples) == 1:
    axes = axes.reshape(1, -1)

for row, img_path in enumerate(demo_samples):
    # Original
    orig = cv2.imread(str(img_path))
    orig_rgb = cv2.cvtColor(orig, cv2.COLOR_BGR2RGB)
    axes[row, 0].imshow(orig_rgb)
    axes[row, 0].set_title("Original", fontsize=12)
    axes[row, 0].axis("off")

    # Gaussian blur
    anon_gauss, n_det_g, dets_g = anonymize_image(
        img_path, best_model, conf=0.25, blur_intensity=51, method="gaussian"
    )
    axes[row, 1].imshow(cv2.cvtColor(anon_gauss, cv2.COLOR_BGR2RGB))
    axes[row, 1].set_title(f"Gaussian Blur ({n_det_g} det)", fontsize=12)
    axes[row, 1].axis("off")

    # Pixelate
    anon_pix, n_det_p, dets_p = anonymize_image(
        img_path, best_model, conf=0.25, method="pixelate"
    )
    axes[row, 2].imshow(cv2.cvtColor(anon_pix, cv2.COLOR_BGR2RGB))
    axes[row, 2].set_title(f"Pixelación ({n_det_p} det)", fontsize=12)
    axes[row, 2].axis("off")

    # Imprimir detecciones
    for d in dets_g:
        print(f"  {img_path.name}: {d['class']} ({d['confidence']:.2f})")

plt.suptitle("Pipeline de Anonimización: Original → Gaussian Blur → Pixelación",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 18. Anonimización en lote y video

Procesamiento de directorios completos y soporte para video con medición de FPS.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 18. Anonimización en lote y video
# ══════════════════════════════════════════════════════════════

def anonymize_directory(
    input_dir: str,
    output_dir: str,
    model: YOLO,
    conf: float = 0.25,
    imgsz: int = 1280,
    blur_intensity: int = 51,
    method: str = "gaussian",
    device: int = 0,
) -> dict:
    """Anonimiza todas las imágenes en un directorio."""
    input_path = Path(input_dir)
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tiff"}
    image_files = [f for f in input_path.iterdir()
                   if f.suffix.lower() in extensions]

    total_detections = 0
    processed = 0
    errors = 0
    times = []

    print(f"📂 Procesando {len(image_files)} imágenes de {input_dir}")
    print(f"📁 Salida: {output_dir}\n")

    for img_path in image_files:
        try:
            t0 = time.time()
            anon_img, n_det, dets = anonymize_image(
                str(img_path), model, conf=conf, imgsz=imgsz,
                blur_intensity=blur_intensity, method=method, device=device
            )
            elapsed = time.time() - t0
            times.append(elapsed)

            out_path = output_path / img_path.name
            cv2.imwrite(str(out_path), anon_img)

            total_detections += n_det
            processed += 1

            if processed % 50 == 0 or processed == len(image_files):
                avg_fps = 1.0 / (sum(times) / len(times)) if times else 0
                print(f"  [{processed}/{len(image_files)}] "
                      f"Detecciones acumuladas: {total_detections} | "
                      f"FPS promedio: {avg_fps:.1f}")

        except Exception as e:
            errors += 1
            print(f"  ⚠️ Error en {img_path.name}: {e}")

    avg_fps = 1.0 / (sum(times) / len(times)) if times else 0
    summary = {
        "processed": processed,
        "errors": errors,
        "total_detections": total_detections,
        "avg_fps": avg_fps,
        "total_time": sum(times),
    }

    print(f"\n{'=' * 50}")
    print(f"✅ Procesamiento completado")
    print(f"   Imágenes: {processed}/{len(image_files)}")
    print(f"   Detecciones: {total_detections}")
    print(f"   FPS promedio: {avg_fps:.1f}")
    print(f"   Tiempo total: {sum(times):.1f}s")
    if errors > 0:
        print(f"   ⚠️ Errores: {errors}")

    return summary


def anonymize_video(
    video_path: str,
    output_path: str,
    model: YOLO,
    conf: float = 0.25,
    imgsz: int = 1280,
    blur_intensity: int = 51,
    method: str = "gaussian",
    device: int = 0,
) -> dict:
    """Anonimiza un video frame por frame."""
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise ValueError(f"No se pudo abrir el video: {video_path}")

    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(str(output_path), fourcc, fps, (width, height))

    frame_count = 0
    total_detections = 0
    times = []

    print(f"🎬 Video: {video_path}")
    print(f"   Resolución: {width}x{height} @ {fps}fps")
    print(f"   Frames totales: {total_frames}\n")

    # Guardar frame temporalmente para usar anonymize_image
    temp_frame_path = Path(output_path).parent / "_temp_frame.jpg"

    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            t0 = time.time()

            # Guardar frame temporal
            cv2.imwrite(str(temp_frame_path), frame)
            anon_frame, n_det, _ = anonymize_image(
                str(temp_frame_path), model, conf=conf, imgsz=imgsz,
                blur_intensity=blur_intensity, method=method, device=device
            )

            out.write(anon_frame)
            elapsed = time.time() - t0
            times.append(elapsed)

            total_detections += n_det
            frame_count += 1

            if frame_count % 100 == 0:
                avg_fps = 1.0 / (sum(times[-100:]) / len(times[-100:]))
                print(f"  Frame {frame_count}/{total_frames} | "
                      f"Det: {total_detections} | FPS: {avg_fps:.1f}")
    finally:
        cap.release()
        out.release()
        if temp_frame_path.exists():
            temp_frame_path.unlink()

    avg_fps = 1.0 / (sum(times) / len(times)) if times else 0
    print(f"\n✅ Video anonimizado: {output_path}")
    print(f"   Frames: {frame_count} | Det: {total_detections} | FPS: {avg_fps:.1f}")

    return {
        "frames": frame_count,
        "total_detections": total_detections,
        "avg_fps": avg_fps,
        "total_time": sum(times),
    }


# ── Demo: procesar directorio de validación ──
demo_output_dir = BASE_DIR / "anonymized_demo"
demo_summary = anonymize_directory(
    input_dir=str(DATASET_DIR / "images" / "val"),
    output_dir=str(demo_output_dir),
    model=best_model,
    conf=0.25,
    blur_intensity=51,
    method="gaussian",
)

print(f"\n📊 Resumen: {demo_summary}")

---
## 19. Exportar modelo y resumen final

Exportar el modelo entrenado, copiar a Google Drive, y generar instrucciones de uso standalone.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 19. Exportar modelo y resumen final
# ══════════════════════════════════════════════════════════════
import shutil

# ── Copiar best.pt a Google Drive ──
best_pt = TRAIN_DIR / "weights" / "best.pt"
last_pt = TRAIN_DIR / "weights" / "last.pt"

drive_export_dir = Path("/content/drive/MyDrive/SIRCCD/models/anonymization")
drive_export_dir.mkdir(parents=True, exist_ok=True)

if best_pt.exists():
    shutil.copy2(str(best_pt), str(drive_export_dir / "best.pt"))
    print(f"✅ best.pt copiado a Google Drive: {drive_export_dir / 'best.pt'}")
    print(f"   Tamaño: {best_pt.stat().st_size / 1e6:.1f} MB")
else:
    print("⚠️ best.pt no encontrado")

if last_pt.exists():
    shutil.copy2(str(last_pt), str(drive_export_dir / "last.pt"))
    print(f"✅ last.pt copiado a Google Drive")

# Copiar data.yaml y métricas
yaml_src = DATASET_DIR / "data.yaml"
if yaml_src.exists():
    shutil.copy2(str(yaml_src), str(drive_export_dir / "data.yaml"))

# Copiar gráficos de resultados
results_png = TRAIN_DIR / "results.png"
if results_png.exists():
    shutil.copy2(str(results_png), str(drive_export_dir / "results.png"))

# ── Copiar directorio completo de entrenamiento ──
drive_runs_dir = drive_export_dir / "train_results"
if TRAIN_DIR.exists():
    shutil.copytree(str(TRAIN_DIR), str(drive_runs_dir), dirs_exist_ok=True)
    print(f"✅ Resultados completos copiados a {drive_runs_dir}")

# ── Resumen final ──
print(f"""
{'═' * 70}
 RESUMEN FINAL — SIRCCD Anonymization Model
{'═' * 70}

 Modelo:           {MODEL_NAME}
 Clases:           {CLASS_NAMES}
 Imagen:           {IMG_SIZE}px
 Epochs:           {EPOCHS}
 Dataset:          {DATASET_DIR}

 📊 Métricas (val):
    Precision:     {box.mp:.4f}
    Recall:        {box.mr:.4f}
    mAP@50:        {box.map50:.4f}
    mAP@50-95:     {box.map:.4f}

 📁 Archivos exportados:
    Google Drive:  {drive_export_dir}
    best.pt:       {drive_export_dir / 'best.pt'}
    data.yaml:     {drive_export_dir / 'data.yaml'}

{'═' * 70}
 USO STANDALONE (después de descargar best.pt):
{'═' * 70}

 from ultralytics import YOLO
 import cv2

 model = YOLO("best.pt")
 results = model.predict("imagen.jpg", conf=0.25, imgsz=1280)

 # Para anonimizar:
 for box in results[0].boxes:
     x1, y1, x2, y2 = map(int, box.xyxy[0])
     img = cv2.imread("imagen.jpg")
     roi = img[y1:y2, x1:x2]
     img[y1:y2, x1:x2] = cv2.GaussianBlur(roi, (51, 51), 0)
     cv2.imwrite("anonimizada.jpg", img)

{'═' * 70}
""")

# ── Limpieza opcional ──
print("🧹 Para limpiar el espacio temporal en Colab, ejecuta:")
print(f"   !rm -rf {BASE_DIR / 'datasets'}")
print(f"   !rm -rf {BASE_DIR / 'runs'}")
print(f"   !rm -rf {BASE_DIR / 'anonymized_demo'}")
print("\n✅ ¡Notebook completado! El modelo está listo en Google Drive.")